In [ ]:
import torch

# Load your model
model = torch.load("model.pt")
model.eval()

# Dummy input (adjust shape to your model)
dummy_input = torch.randn(1, 3, 224, 224)

# Export to ONNX
torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=11
)

In [ ]:
import onnx

model = onnx.load("model.onnx")
onnx.checker.check_model(model)

In [ ]:
import tensorrt as trt

logger = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(logger)
network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
parser = trt.OnnxParser(network, logger)

with open("model.onnx", "rb") as f:
    parser.parse(f.read())

config = builder.create_builder_config()
config.max_workspace_size = 1 << 30  # 1GB

engine = builder.build_engine(network, config)

with open("model.engine", "wb") as f:
    f.write(engine.serialize())